# FPL Bot v2 — Comprehensive Training & Team Builder

**Improvements over v1:**
- Trains on **6 complete seasons** (2020-21 to 2025-26) giving ~180k rows
- **Vaastav understat xG** (shot-level npxG/90) integrated as extra features
- **ESPN news headlines** for supplementary player context (no API key)
- **Evaluation vs average human manager** using FPL API `average_entry_score`
- All factors modelled: goals, assists, clean sheets, salary, form, injuries, set-pieces
- Final **squad recommendation ready for 2026/27 GW1**

Training target: 2026/27 (26/27 results NOT in training — no look-ahead bias).

| Section | Content |
|---|---|
| 1 | Data: 6 seasons + ESPN + understat |
| 2 | Understat xG enrichment |
| 3 | Feature engineering |
| 4 | ML training (6 seasons) |
| 5 | Walk-forward backtest + human comparison |
| 6 | Monte Carlo simulation |
| 7 | Final squad optimisation |
| 8 | RL chip agent |
| 9 | Summary & known issues |

## Setup
Installs the `bot` package directly from GitHub. **Requires the repo to be public.**
Run this once per Colab session — restart runtime if packages were already cached.


In [ ]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/DH4410/fpl-auto.git'],
        check=True
    )

print('setup done')


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bot import data_collector as dc
from bot import feature_engineering as fe
from bot import models as M
from bot import simulator as S
from bot import optimizer as O
from bot import rl_agent as R
from bot.fpl_rules import RULES, GKP, DEF, MID, FWD, POSITION_NAMES

try:
    from bot import news_collector as NC
    _HAS_NEWS = True
except ImportError:
    NC = None
    _HAS_NEWS = False
    print('WARNING: news_collector.py missing from bot/ — ESPN enrichment disabled.')
    print('Fix: add bot/news_collector.py to your Colab folder and re-run.')

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

print('modules loaded')
print('default training seasons:', dc.DEFAULT_TRAINING_SEASONS)
print('news_collector available:', _HAS_NEWS)


---
## 1. Data collection & inspection

**Sources (all free, no API keys):**
- FPL API: 564 players, 20 teams, 38 gameweeks for 2026/27
- Vaastav merged_gw.csv: 6 seasons of per-gameweek player rows
- Vaastav understat: shot-level xG/xA per season (npxG, xGChain)
- football-data.co.uk: Pinnacle closing odds for Dixon-Coles ratings
- ESPN public API: latest PL news headlines (no auth)
- FPL set-piece notes: penalty / corner takers

In [ ]:
# 1a. Current 2026/27 FPL data
bs = dc.bootstrap_frames()
players, teams, events = bs["players"], bs["teams"], bs["events"]
print(f"Players: {len(players)}   Teams: {len(teams)}   Gameweeks: {len(events)}")
print("Next GW:", dc.current_gameweek())

pool = dc.build_player_pool()
print(f"Player pool: {len(pool)} total, {pool['available'].sum()} available")
print(f"Price range: £{pool['price'].min():.1f}m – £{pool['price'].max():.1f}m")
pool[["web_name", "team_short", "element_type", "price", "status",
      "selected_by_percent", "chance_of_playing_next_round"]].head(10)

In [ ]:
# 1b. Historical training data — 6 seasons (NOT including 2026/27)
history = dc.load_multi_season_history()   # uses DEFAULT_TRAINING_SEASONS

print(f"Total training rows: {len(history):,}")
print("\nPer-season coverage:")
for season, cov in history.attrs.get("coverage", {}).items():
    dc_str = "YES" if cov.get("has_defcon") else "NONE (pre-DC rule)"
    print(f"  {season}: {cov['rows']:>7,} rows | DC columns: {dc_str}")

print("\nKey column availability:")
key_cols = ["goals_scored", "assists", "clean_sheets", "expected_goals",
            "expected_assists", "clearances_blocks_interceptions"]
for col in key_cols:
    pct = history[col].notna().mean() * 100 if col in history.columns else 0
    print(f"  {col:<42} {pct:.1f}% non-null")

In [ ]:
# 1c. Historical match odds (Pinnacle closing prices)
odds_seasons = {}
for s in ("2020-21", "2021-22", "2022-23", "2023-24", "2024-25"):
    try:
        odds_seasons[s] = dc.fetch_football_data_odds(s)
        print(f"  {s}: {len(odds_seasons[s])} matches — "
              f"{odds_seasons[s].attrs.get('odds_source')}")
    except dc.DataFetchError as e:
        print(f"  {s}: skipped — {e}")

odds_2425 = odds_seasons.get("2024-25", pd.DataFrame())
print(f"\nDixon-Coles training: {len(odds_2425)} matches from 2024-25")

In [ ]:
# 1d. Understat shot-level xG (from Vaastav's understat/ directory)
understat = dc.load_understat_history()
if not understat.empty:
    print(f"Understat rows: {len(understat):,} ({understat['season'].nunique()} seasons)")
    print("Columns:", list(understat.columns[:12]))
    if "xG" in understat.columns and "player" in understat.columns:
        top_xg = (understat.groupby("player")["xG"].sum()
                  .sort_values(ascending=False).head(10).reset_index())
        print("\nTop 10 by total xG (all seasons):")
        print(top_xg.to_string(index=False))
else:
    print("Understat unavailable — will use FPL expected_goals instead")

In [ ]:
# 1e. ESPN PL news + FPL official injury flags
espn_news = dc.fetch_espn_news(limit=50)
print(f"ESPN headlines: {len(espn_news)}")
for a in espn_news[:5]:
    print(f"  [{a.get('published','')[:10]}] {a.get('headline','')}")

fpl_news_players = pool[pool["news"].astype(str).str.len() > 3].copy()
print(f"\nFPL injury/news flags: {len(fpl_news_players)} players")
fpl_news_players[["web_name", "team_short", "status", "news",
                   "chance_of_playing_next_round"]].head(12)

In [ ]:
# 1f. Set-piece notes (penalty + corner + free-kick takers)
sp = dc.fetch_set_piece_notes()
teams_sp = [t for t in sp.get("teams", []) if t.get("notes")]
print(f"Set-piece notes for {len(teams_sp)} teams:")
for t in teams_sp[:6]:
    print(f"  {t.get('teamName','')}: {t.get('notes','')[:120]}")

---
## 2. Understat xG enrichment

Vaastav mirrors understat.com per season. We compute npxG/90, xGChain/90 and
join to the training history by player name + season. These features expose
true attacking quality independent of noisy actual-goal counts.

In [ ]:
def merge_understat_to_history(history_df, understat_df):
    """Join understat per-90 rates onto history by (player_norm, season)."""
    if understat_df.empty:
        print("No understat data — skipping"); return history_df
    if "player" not in understat_df.columns or "season" not in understat_df.columns:
        print("Understat missing columns — skipping"); return history_df

    us = understat_df.copy()
    us["player_norm"] = us["player"].str.lower().str.strip()
    hist = history_df.copy()
    name_col = "name" if "name" in hist.columns else None
    if name_col is None:
        print("History missing 'name' column — skipping understat"); return hist
    hist["player_norm"] = hist[name_col].str.lower().str.strip()

    rate_cols = [c for c in ["xG", "xA", "npxG", "xGChain", "xGBuildup"] if c in us.columns]
    if "time" in us.columns:
        for c in rate_cols:
            us[f"{c}_per90"] = us[c] / us["time"].clip(1) * 90

    per90 = [f"{c}_per90" for c in rate_cols if f"{c}_per90" in us.columns]
    if not per90:
        print("No per90 cols available — skipping"); return hist

    agg = us.groupby(["player_norm", "season"])[per90].mean().reset_index()
    merged = hist.merge(agg, on=["player_norm", "season"], how="left")
    n = merged[per90[0]].notna().sum()
    print(f"Understat join: {n:,}/{len(merged):,} rows matched ({100*n/len(merged):.1f}%)")
    return merged

hist_with_us = merge_understat_to_history(history, understat)
us_extra = [c for c in hist_with_us.columns if "_per90" in c and
            c not in ("defcon_per90",)]
print("Extra understat features:", us_extra)

---
## 3. Feature engineering

Five feature families (6-season data):
1. **EWMA form** (alpha=0.25) — shifted 1 GW to prevent leakage
2. **Rolling windows** (3 GW + 6 GW) — catches sudden form breaks
3. **Dixon-Coles ratings** — attack/defence per team, MLE + time decay
4. **De-vigged odds** (Shin's method) — Pinnacle prices → goal rates
5. **News availability** — FPL news sentiment + ESPN headline sentiment

In [ ]:
# 3a. EWMA form (shifted one GW — no leakage)
feat = fe.compute_ewma_stats(hist_with_us, alpha=0.25)
feat = fe.rolling_window_stats(feat)

feat["element_type"] = feat["position"].map(
    {"GK": GKP, "GKP": GKP, "DEF": DEF, "MID": MID, "FWD": FWD}
)
for pid, label in ((GKP, "gkp"), (DEF, "def"), (MID, "mid"), (FWD, "fwd")):
    feat[f"pos_{label}"] = (feat["element_type"] == pid).astype(float)

feature_cols = fe.feature_columns(feat)
print(f"{len(feature_cols)} model features")

first_rows = feat.groupby("element").head(1)
nan_frac = first_rows["ewma_minutes"].isna().mean()
print(f"First-appearance NaN EWMA: {nan_frac:.2%} (must be 1.00 — leakage check)")
assert nan_frac == 1.0, "LEAKAGE DETECTED!"
print("Leakage check passed.")

In [ ]:
# 3b. Dixon-Coles team ratings (fit on names, not FPL ids)
strength = fe.fit_team_strength_by_name(odds_2425, teams)
strength = fe.promoted_team_priors(teams, strength, quantile=0.25)
print(f"Home advantage: {strength.attrs.get('home_advantage', float('nan')):.3f}")
print(f"Prior-seeded clubs (promoted): {int(strength['is_prior'].sum())}")

strength.sort_values("attack", ascending=False)[
    ["team_name", "attack", "defence",
     "expected_scored", "expected_conceded", "is_prior"]
].round(3).head(10)

In [ ]:
# 3c. News / availability enrichment
if _HAS_NEWS:
    try:
        pool_enriched = NC.build_news_features(pool, espn_enriched=True)
        n_espn = pool_enriched.get('espn_id', pd.Series()).notna().sum()
        print(f'ESPN athletes matched: {n_espn}')
    except Exception as exc:
        print(f'ESPN enrichment skipped ({exc}); FPL news only')
        pool_enriched = NC.build_news_features(pool, espn_enriched=False)
else:
    pool_enriched = pool.copy()
    pool_enriched['news_sentiment'] = 0.0
    pool_enriched['avail_next'] = pool_enriched.get(
        'chance_of_playing_next_round', pd.Series(100, index=pool_enriched.index)
    ).fillna(100) / 100.0
    pool_enriched['availability_index'] = pool_enriched['avail_next']
    print('ESPN enrichment skipped (news_collector missing) — using FPL status only')

if _HAS_NEWS:
    avail = fe.news_sentiment_features(pool_enriched, dc.fetch_set_piece_notes())
else:
    avail = pool_enriched
flagged = avail[avail['news'].astype(str).str.len() > 3]
print(f'{len(flagged)} players with FPL news notes')
if not flagged.empty:
    cols = [c for c in ['web_name','team_short','status','news',
                         'news_sentiment','avail_next','availability_index']
            if c in flagged.columns]
    print(flagged.nsmallest(10, 'availability_index')[cols].to_string(index=False))


---
## 4. ML model training — 6 seasons (~180k rows)

| Model | Target | Algorithm | Seasons |
|---|---|---|---|
| MinutesModel | minutes 0-90 | XGBoost (asym. loss, over-predict penalised 3x) | 2020-26 |
| AttackModel | xG/90, xA/90 | LightGBM x2 | 2020-26 |
| DefenseModel | CS prob + DC/90 | XGBoost | CS: 2020-26; DC: 2025-26 |
| BonusModel | BPS / bonus | XGBoost + empirical curve | 2020-26 |

In [ ]:
# 4a. Build training matrix
targets = M.build_training_targets(feat)
X_all = feat[feature_cols].astype(float)
X_all = X_all.fillna(X_all.median())

valid = targets["minutes"].notna().to_numpy()
X = X_all[valid].reset_index(drop=True)
y = {k: pd.Series(np.asarray(v))[valid].reset_index(drop=True)
     for k, v in targets.items()}
element_types = feat["element_type"].to_numpy()[valid]

print(f"Training rows (6 seasons): {len(X):,}  ({len(X)/57000:.1f}x v1)")
print(f"Features: {len(feature_cols)}")
print(f"DC-labelled rows: {int(y['defcon_per90'].notna().sum()):,}")
print("\nSeason breakdown:")
for s, grp in feat.groupby("season"):
    cnt = int(grp["total_points"].notna().sum())
    print(f"  {s}: {cnt:>7,} rows")

In [ ]:
# 4b. Train all sub-models
predictor = M.FPLPointsPredictor()

predictor.minutes_model.train(X, y["minutes"])
print("✓ MinutesModel")

rate_rows = y["xg_per90"].notna().to_numpy()
predictor.attack_model.train(
    X[rate_rows], y["xg_per90"][rate_rows], y["xa_per90"][rate_rows]
)
print(f"✓ AttackModel  ({rate_rows.sum():,} rows with xG)")

predictor.defense_model.calibrate_position_priors(feat)
predictor.defense_model.train(X, y["clean_sheet"], y["defcon_per90"])
print(f"✓ DefenseModel  DC rule-based: {predictor.defense_model.defcon_is_rule_based}")

predictor.bonus_model.train(X, y["bps"], y["bonus"])
print("✓ BonusModel\n\nAll models trained.")

In [ ]:
# 4c. Feature importances
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, model) in zip(axes, [
        ("Minutes", predictor.minutes_model),
        ("Attack (xG/90)", predictor.attack_model),
        ("Bonus (BPS)", predictor.bonus_model)]):
    imp = model.feature_importances(12)
    ax.barh(imp["feature"][::-1], imp["importance"][::-1], color="#3b82f6")
    ax.set_title(f"{title} — top 12", fontsize=11)
    ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

---
## 5. Walk-forward backtest & human manager comparison

At gameweek *t*: train on everything before *t*, evaluate on *t* alone.
k-fold is invalid here (shuffles rows → leaks future data).

**Human baseline:** FPL `average_entry_score` per GW = average of ~12M managers.
Target: consistently beat 45 pts/GW (typical average manager score).

In [ ]:
# 5a. Walk-forward validation
backtest_frame = feat.copy()
for col in feature_cols:
    backtest_frame[col] = X_all[col]

print("Walk-forward minutes (may take 1-3 min)...")
wf_minutes = M.walk_forward_validation(
    backtest_frame, feature_cols, "minutes",
    lambda: M.MinutesModel(n_estimators=150),
    min_train_periods=8, step=3
)

print("Walk-forward points...")
wf_points = M.walk_forward_validation(
    backtest_frame, feature_cols, "total_points",
    lambda: M.BonusModel(n_estimators=150),
    min_train_periods=8, step=3
)

print("\n── v2 results ──────────────────────────")
print(f"  Minutes MAE:       {wf_minutes['mae'].mean():.2f}  (v1: 14.72)")
print(f"  Minutes Spearman:  {wf_minutes['spearman'].mean():.3f}  (v1: 0.755)")
print(f"  Minutes bias:      {wf_minutes['bias'].mean():.2f}  (negative=conservative)")
print(f"  Points MAE:        {wf_points['mae'].mean():.3f}  (v1: 1.75)")
print(f"  Points Spearman:   {wf_points['spearman'].mean():.3f}  (v1: 0.680)")

In [ ]:
# 5b. Walk-forward plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(wf_minutes["mae"], marker="o", color="#3b82f6", label="v2")
axes[0].axhline(14.72, color="gray", linestyle="--", alpha=0.7, label="v1 (14.72)")
axes[0].set_title("Minutes MAE by period"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(wf_minutes["spearman"], marker="o", color="#3b82f6", label="minutes ρ (v2)")
axes[1].plot(wf_points["spearman"], marker="s", color="#ef4444", label="points ρ (v2)")
axes[1].axhline(0.755, color="#93c5fd", linestyle="--", alpha=0.7, label="minutes v1")
axes[1].axhline(0.680, color="#fca5a5", linestyle="--", alpha=0.7, label="points v1")
axes[1].set_ylim(0, 1); axes[1].set_title("Spearman ρ")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 5c. Human manager comparison
if "average_entry_score" in events.columns:
    human = events[events["average_entry_score"].notna()].copy()
    avg = human["average_entry_score"].mean()
    peak = human["highest_score"].max() if "highest_score" in human.columns else None
    print(f"Average manager score per GW: {avg:.1f} pts")
    if peak: print(f"Best single GW score: {peak:.0f} pts")
    print()
    print("2026/27 hasn't started — GW-by-GW comparison available after GW1.")
    print(f"Points Spearman {wf_points['spearman'].mean():.3f} should beat the")
    print(f"average manager ({avg:.0f} pts/GW) consistently over a full season.")
else:
    print("No average_entry_score yet (preseason).")
    print(f"Walk-forward Spearman {wf_points['spearman'].mean():.3f} is the best proxy.")

---
## 6. Monte Carlo simulation — GW1 2026/27

50,000 simulations per player. Captaincy needs the full distribution:
double-pointing a player with P(haul>=15)=30% beats one with lower upside.

In [ ]:
# 6a. Bivariate Poisson fitted on 2024-25 odds
name_map = dc.match_team_names(
    pd.unique(pd.concat([odds_2425["HomeTeam"], odds_2425["AwayTeam"]])), teams
)
results = odds_2425.rename(columns={"HomeTeam": "home_team", "AwayTeam": "away_team"}).copy()
results["home_team"] = results["home_team"].map(name_map)
results["away_team"] = results["away_team"].map(name_map)
results = results.dropna(subset=["home_team", "away_team"])

match_sim = S.BivariatePoissonSimulator(strength=strength.copy())
match_sim.fit(results)
print(f"lambda3={match_sim.lambda3:.4f}  home_advantage={match_sim.home_advantage:.3f}")
print("Note: lambda3≈0 correct — real scorelines are mildly negatively dependent")

In [ ]:
# 6b. Per-player rate table — uses FPL element-summary to fix the 2026/27 ID reset
#
# ROOT CAUSE (fixed here): FPL reset all player element IDs in 2026/27
# (Raya=1, Saka=12, etc.). Matching current IDs against Vaastav historical
# data therefore maps every player to a WRONG historical player's features.
# /api/element-summary/{id}/history_past is keyed to the CURRENT element ID
# and contains full season-by-season stats — no ID mismatch possible.

current = pool_enriched[pool_enriched["available"]].copy().reset_index(drop=True)
print(f"Available players: {len(current)}")

# Fetch element-summary for all available players (cached 24h — fast on re-run)
print("Fetching FPL element summaries (first run ~60-90s, cached after)...")
summaries = dc.fetch_element_summaries_bulk(current["id"].tolist())

fpl_hist = dc.element_summaries_to_features(summaries, recent_seasons=2)
print(f"history_past features: {len(fpl_hist)} players  "
      f"({len(current) - len(fpl_hist)} new/missing — will get positional prior)")

# Align to current player list; any player absent from history_past is new this season
aligned = fpl_hist.reindex(current["id"].values)

# For players with no FPL history (very new signings, promoted-team debutants),
# fall back to FPL API bootstrap stats converted to per-game rates
fpl_src = pool_enriched.set_index("id")
games = (fpl_src["starts"].reindex(current["id"]).fillna(0)
         if "starts" in fpl_src.columns
         else pd.Series(38.0, index=current["id"]))
games = games.clip(lower=1)

FPL_BOOTSTRAP_TO_EWMA = {
    "expected_goals":             "ewma_expected_goals",
    "expected_assists":           "ewma_expected_assists",
    "expected_goal_involvements": "ewma_expected_goal_involvements",
    "expected_goals_conceded":    "ewma_expected_goals_conceded",
    "goals_scored":               "ewma_goals_scored",
    "assists":                    "ewma_assists",
    "clean_sheets":               "ewma_clean_sheets",
    "bonus":                      "ewma_bonus",
    "minutes":                    "ewma_minutes",
    "influence":                  "ewma_influence",
    "creativity":                 "ewma_creativity",
    "threat":                     "ewma_threat",
    "ict_index":                  "ewma_ict_index",
    "total_points":               "ewma_total_points",
    "saves":                      "ewma_saves",
    "goals_conceded":             "ewma_goals_conceded",
}
no_hist = aligned.isna().all(axis=1)
for fpl_col, ewma_col in FPL_BOOTSTRAP_TO_EWMA.items():
    if fpl_col not in fpl_src.columns or ewma_col not in aligned.columns:
        continue
    per_game = fpl_src[fpl_col].reindex(current["id"]).fillna(0) / games
    aligned.loc[no_hist, ewma_col] = per_game.loc[no_hist].values

print(f"  from history_past: {(~no_hist).sum()}  "
      f"from bootstrap fallback: {no_hist.sum()}")

X_now = aligned[feature_cols].astype(float).fillna(X_all.median()).reset_index(drop=True)

live = predictor.predict(X_now, current["element_type"].to_numpy())
squad_stats = pd.DataFrame({
    "element":          current["id"].to_numpy(),
    "team":             current["team"].to_numpy(),
    "element_type":     current["element_type"].to_numpy(),
    "xg_per90":         live["xg_per90"].to_numpy(),
    "xa_per90":         live["xa_per90"].to_numpy(),
    "expected_minutes": live["expected_minutes"].to_numpy(),
    "p60":              live["p60"].to_numpy(),
    "defcon_per90":     live["defcon_per90"].to_numpy(),
})
print(f"Available for simulation: {len(squad_stats)} players")


In [ ]:
# 6b-diagnostic: Why are certain premium players absent?
# -----------------------------------------------------------
# Run this after 6b to understand the model's reasoning.
WATCH = ["Haaland", "Salah", "Raya", "Saka", "Palmer", "Watkins", "De Bruyne", "Alexander-Arnold"]

print("=== 1. AVAILABILITY (FPL status) ===")
for name in WATCH:
    m = pool[pool["web_name"].str.contains(name, case=False, na=False)]
    if m.empty:
        print(f"  {name:22s}  NOT in FPL pool at all")
    else:
        for _, r in m.iterrows():
            avail = "AVAILABLE" if r.get("available") else f"EXCLUDED  status={r.get('status')!r}"
            print(f"  {r['web_name']:22s}  {r['team_short']:4s}  £{r.get('price',0):.1f}m  {avail}  ppg={r.get('points_per_game',0):.2f}")

print("\n=== 2. HISTORICAL FEATURE QUALITY ===")
print(f"  Players in Vaastav history : {aligned.notna().any(axis=1).sum()}/{len(aligned)}")
print(f"  Fully missing → got median : {aligned.isna().all(axis=1).sum()}")
print()
for name in WATCH:
    m = pool[pool["web_name"].str.contains(name, case=False, na=False)]
    for _, r in m.iterrows():
        pid = r["id"]
        if pid not in aligned.index:
            print(f"  {r['web_name']:22s}  ID {pid} absent from hist index entirely")
            continue
        row = aligned.loc[pid]
        all_nan = row.isna().all()
        xg   = row.get("ewma_expected_goals", float("nan"))
        mins = row.get("ewma_minutes", float("nan"))
        pts  = row.get("ewma_total_points", float("nan"))
        src  = "MEDIAN FALLBACK (no history)" if all_nan else f"ewma_xG={xg:.3f}  ewma_mins={mins:.1f}  ewma_pts={pts:.2f}"
        print(f"  {r['web_name']:22s}  {src}")

print("\n=== 3. SIMULATION SCORES (GW1 mean) ===")
for name in WATCH:
    m = pool[pool["web_name"].str.contains(name, case=False, na=False)]
    for _, r in m.iterrows():
        pid = r["id"]
        if pid in summary.index:
            s = summary.loc[pid]
            print(f"  {r['web_name']:22s}  mean={s['mean']:.2f}  p80={s['p80']:.0f}  p_haul={s.get('p_haul',0):.2f}  rank={int((summary['mean'] > s['mean']).sum())+1}")
        else:
            avail = r.get("available", False)
            print(f"  {r['web_name']:22s}  NOT SIMULATED (available={avail})")

print("\n=== 4. GW1 FIXTURES ===")
_tname = teams.set_index("id")["name"].to_dict()
_tshort = teams.set_index("id")["short_name"].to_dict()
for _, fx in gw_fixtures.iterrows():
    h, a = _tname.get(fx.team_h, fx.team_h), _tname.get(fx.team_a, fx.team_a)
    print(f"  {h:25s} vs {a}")


In [ ]:
# 6c. Simulate GW1
fixtures = dc.fixtures_frame()
gw = dc.current_gameweek() or 1
gw_fixtures = fixtures[fixtures["event"] == gw]
print(f"GW{gw}: {len(gw_fixtures)} fixtures, 50k sims")

season_sim = S.SeasonSimulator(match_sim=match_sim)
summary, samples = season_sim.simulate_gameweek(
    gw_fixtures, squad_stats, n_sims=50_000, return_samples=True
)

extra = [c for c in ["web_name", "team_short", "price", "element_type"]
         if c in pool_enriched.columns]
named = summary.join(pool_enriched.set_index("id")[extra])
named["pos"] = named["element_type"].map(POSITION_NAMES)

print(f"\nTop 20 projected — GW{gw}:")
named.sort_values("mean", ascending=False).head(20)[
    ["web_name", "team_short", "pos", "price",
     "mean", "std", "p80", "p90", "p_haul", "p_blank"]
].round(2)

In [ ]:
# 6d. Captaincy distributions
top_ids = named.sort_values("mean", ascending=False).head(5).index.tolist()
positions = {e: i for i, e in enumerate(squad_stats["element"])}

fig, ax = plt.subplots(figsize=(13, 5))
colors = ["#3b82f6", "#ef4444", "#10b981", "#f59e0b", "#8b5cf6"]
for element, color in zip(top_ids, colors):
    if element not in positions: continue
    draws = samples[positions[element]]
    n = named.loc[element, "web_name"]
    ax.hist(draws, bins=np.arange(0, 26) - 0.5, alpha=0.5, density=True,
            color=color, label=f"{n}  μ={draws.mean():.1f}")
ax.set_xlabel("GW points"); ax.set_ylabel("Probability")
ax.set_title(f"GW{gw} distributions — captaincy candidates")
ax.legend(); plt.tight_layout(); plt.show()

cap = season_sim.captaincy_analysis(
    summary, samples, squad_stats["element"].to_numpy(), top_n=8
)
cap.join(named[["web_name", "team_short", "pos", "price"]]).round(3).head(8)

---
## 7. Final squad optimisation — 2026/27

MIP (HiGHS / CBC) under: £100m, 2/5/5/3 by position,
max 3/club, valid formation, availability-weighted pts.

In [ ]:
# 7b. Full squad display with GW1 fixture context
squad_df = pd.DataFrame(best["squad"])
squad_df["pos"] = squad_df["position"].map(POSITION_NAMES)
xi_els = {p["element"] for p in best["starting_xi"]}
cap_el, vc_el = best.get("captain"), best.get("vice_captain")
squad_df["role"] = squad_df.apply(
    lambda r: ("XI (C)" if r["element"] == cap_el
               else "XI (VC)" if r["element"] == vc_el
               else "XI" if r["element"] in xi_els
               else "bench"),
    axis=1
)

# Build a GW1 fixture label per player (home/away vs opponent)
_tshort_map = teams.set_index("id")["short_name"].to_dict()
_player_team = current.set_index("id")["team"].to_dict()
_fx_home = {}  # team_id → (opponent_short, is_home)
for _, fx in gw_fixtures.iterrows():
    _fx_home[int(fx.team_h)] = (_tshort_map.get(int(fx.team_a), "?"), True)
    _fx_home[int(fx.team_a)] = (_tshort_map.get(int(fx.team_h), "?"), False)

def _fixture_label(element):
    team = _player_team.get(int(element))
    if team is None or int(team) not in _fx_home:
        return "BGW"
    opp, is_home = _fx_home[int(team)]
    return f"{'H' if is_home else 'A'} {opp}"

squad_df["gw1"] = squad_df["element"].apply(_fixture_label)

print("=== RECOMMENDED 2026/27 SQUAD ===")
out = squad_df.sort_values(["position", "expected_points"], ascending=[True, False])[
    ["name", "pos", "price", "expected_points", "gw1", "role"]
]
print(out.to_string(index=False))
print(f"\nTotal cost: £{best['total_cost']:.1f}m / £100.0m budget")
print(f"Expected XI pts (GW1, captain doubled): {best['expected_points']:.1f}")


In [ ]:
# 7b. Full squad display
squad_df = pd.DataFrame(best["squad"])
squad_df["pos"] = squad_df["position"].map(POSITION_NAMES)
xi_els = {p["element"] for p in best["starting_xi"]}
cap_el, vc_el = best.get("captain"), best.get("vice_captain")
squad_df["role"] = squad_df.apply(
    lambda r: ("XI (C)" if r["element"] == cap_el
               else "XI (VC)" if r["element"] == vc_el
               else "XI" if r["element"] in xi_els
               else "bench"),
    axis=1
)

print("=== RECOMMENDED 2026/27 SQUAD ===")
out = squad_df.sort_values(["position", "expected_points"], ascending=[True, False])[
    ["name", "pos", "price", "expected_points", "role"]
]
print(out.to_string(index=False))

In [ ]:
# 7c. Squad legality verification
check = squad_df.rename(columns={"position": "element_type"}).copy()
check["now_cost"] = (check["price"] * 10).round().astype(int)
valid_sq, reason = RULES.validate_squad(check.to_dict("records"), budget=100.0)
print(f"Squad legal:      {valid_sq} | {reason}")
xi_pos = [{"element_type": p["position"]} for p in best["starting_xi"]]
print(f"Formation legal:  {RULES.validate_formation(xi_pos)}")
print(f"Max players/club: {check['team'].value_counts().max()} (limit 3)")
print(f"Composition: {check['element_type'].map(POSITION_NAMES).value_counts().to_dict()}")

In [ ]:
# 7d. Best-value players (pts per £m) by position
named_avail = named[named.index.isin(current["id"])].copy()
named_avail["price"] = current.set_index("id")["price"].reindex(named_avail.index)
named_avail["value"] = named_avail["mean"] / named_avail["price"].clip(0.1)
named_avail["avail"] = avail_idx.reindex(named_avail.index).fillna(1.0)

avail_mask = named_avail["avail"] >= 0.6
top_val = (named_avail[avail_mask]
           .groupby("pos", group_keys=False)
           .apply(lambda g: g.nlargest(5, "value")))
print("Best value players per position (availability >= 60%):")
top_val[["web_name", "pos", "price", "mean", "p80", "value"]].round(3)

---
## 8. RL chip agent

Chip timing = Markov decision process. PPO over 38-GW season episodes.

Benchmarks: no chips (lower bound), hand-written heuristic (hold for doubles).

In [ ]:
N_EPISODES = 300   # raise to 5000+ for a real training run

replay = R.SeasonReplayGenerator(history_df=history)
env    = R.FPLEnv(replay=replay, seed=0)
print(f"Obs: {env.observation_space.shape}   Actions: {env.action_space.n}")
print(f"Actions: {R.ACTIONS}")

def run_policy(fn, n=200, seed_off=5000):
    totals = []
    for ep in range(n):
        obs, _ = env.reset(seed=seed_off + ep)
        done, total = False, 0.0
        while not done:
            obs, r, done, _, _ = env.step(fn(env))
            total += r
        totals.append(total)
    return float(np.mean(totals))

def heuristic(e):
    s = {"gameweek": e.gameweek, "chips_available": e.chips_available,
         "n_fixtures": e.season["n_fixtures"][e.gameweek - 1]}
    chip = R.baseline_chip_policy(s)
    return R.ACTIONS.index(chip) if chip else 0

no_chips = run_policy(lambda e: 0)
baseline = run_policy(heuristic)
print(f"\nNo chips:  {no_chips:.0f} pts/season")
print(f"Heuristic: {baseline:.0f} pts/season  (+{baseline - no_chips:.0f})")

In [ ]:
try:
    agent = R.FPLChipAgent(replay=replay)
    agent.train(n_episodes=N_EPISODES)
    log = agent.evaluate(n_episodes=100)
    seasons = log[log["chip"] == "_total"]["reward_gain"]
    print(f"PPO agent:    {seasons.mean():.0f} pts/season")
    print(f"vs heuristic: {seasons.mean() - baseline:+.0f}")
    print(f"vs no chips:  {seasons.mean() - no_chips:+.0f}")
    timing = log[log["chip"] != "_total"].groupby("chip")["gameweek"].agg(
        ["mean", "std", "count"])
    print("\nChip timing (mean ± std GW):")
    print(timing.round(1))
    agent.save("bot/cache/chip_agent")
except ImportError as exc:
    print(f"stable-baselines3 not installed: {exc}")
    print("pip install stable-baselines3  (~2 GB; runs on Colab T4)")

---
## 9. Summary & known issues

In [ ]:
print(f"""
FPL BOT v2 — SUMMARY
=================================================================
TRAINING DATA
  Seasons         : {', '.join(str(s) for s in dc.DEFAULT_TRAINING_SEASONS)}
  Total rows      : {len(X):,}  (v1: ~57,000;  {len(X)/57000:.1f}x more)
  Features        : {len(feature_cols)}
  DC-labelled     : {int(y['defcon_per90'].notna().sum()):,}  (2025-26 only)

BACKTEST (walk-forward, expanding window, min 8 training periods)
  Minutes MAE     : {wf_minutes['mae'].mean():.2f}  (v1: 14.72)
  Minutes Spearman: {wf_minutes['spearman'].mean():.3f}  (v1: 0.755)
  Points Spearman : {wf_points['spearman'].mean():.3f}  (v1: 0.680)
  Minutes bias    : {wf_minutes['bias'].mean():.2f}  (negative = conservative)

RECOMMENDED SQUAD
  Cost            : £{best['total_cost']:.1f}m
  Expected XI pts : {best['expected_points']:.1f}
  Captain         : {best['captain_name']}

DATA SOURCES (all free, zero API keys)
  FPL API         : bootstrap-static, fixtures, element-summary, set-piece-notes
  Vaastav         : merged_gw.csv (6 seasons), understat xG
  football-data   : Pinnacle closing odds for Dixon-Coles
  ESPN (public)   : PL news headlines (no auth)

KNOWN ISSUES
  1  No 2026/27 GW data — season hasn't started. Wide early error bars.
  2  DC has 1 season of history (2025-26). Positional priors for others.
  3  Bonus approximated per-player, not full 22-player BPS simulation.
  4  ESPN soccer injury endpoint returns empty (verified). FPL news field
     is the primary injury signal; ESPN gives headline context only.
  5  RL agent trains on simulated data — compare vs heuristic first.
  6  No effective-ownership objective. Maximises points, not rank.

HOW TO BEAT THE AVERAGE MANAGER
  Target >45 pts/GW. A Spearman of {wf_points['spearman'].mean():.3f} on player rankings
  should achieve this over 38 gameweeks consistently.
  Key edge: chip timing (3-chip DGW/BGW combo) + premium captain rotation.

BOT NEVER EXECUTES TRANSFERS — advisory only.
=================================================================
""")